# SynthQuant Stage 1 VQ-VAE Training on Kaggle

Trains the Stage 1 VQ-VAE compressor for the SynthQuant unified model.

**Config**: chunk_size=8, latent_dim=128, codebook_size=256, PQ subspaces=4

**Datasets**: Streamed from Hugging Face (WikiText, C4, FineWeb, Pile)

**Runtime**: ~2-3 hours on T4 GPU for 400 epochs

In [ ]:
# ── Streaming dataset selection ────────────────────────────────
# Choose ONE dataset source:

# Option 1: WikiText-103 (100M tokens, clean Wikipedia)
# ds = load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1', split='train', streaming=True)

# Option 2: C4 English (750GB Common Crawl, deduplicated)
# ds = load_dataset('allenai/c4', 'en', split='train', streaming=True)

# Option 3: FineWeb (45TB filtered web, highest quality)
# ds = load_dataset('HuggingFaceFW/fineweb', split='train', streaming=True)

# Option 4: The Pile (825GB diverse corpus)
# ds = load_dataset('EleutherAI/pile', split='train', streaming=True)

# Option 5: FineWeb-Edu (educational subset, 1.3T tokens)
# ds = load_dataset('HuggingFaceFW/fineweb-edu', split='train', streaming=True)

# Option 6: Wikipedia (current snapshot)
# ds = load_dataset('wikimedia/wikipedia', '20231101.en', split='train', streaming=True)

from datasets import load_dataset

# ─── SELECT YOUR DATASET HERE ───
DATASET_CHOICE = "wikitext"  # Options: wikitext, c4, fineweb, pile, fineweb-edu, wikipedia

if DATASET_CHOICE == "wikitext":
    ds = load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1', split='train', streaming=True)
    MAX_CHARS = 5_000_000  # ~5M chars from 100M token corpus
elif DATASET_CHOICE == "c4":
    ds = load_dataset('allenai/c4', 'en', split='train', streaming=True)
    MAX_CHARS = 10_000_000
elif DATASET_CHOICE == "fineweb":
    ds = load_dataset('HuggingFaceFW/fineweb', split='train', streaming=True)
    MAX_CHARS = 20_000_000
elif DATASET_CHOICE == "pile":
    ds = load_dataset('EleutherAI/pile', split='train', streaming=True)
    MAX_CHARS = 10_000_000
elif DATASET_CHOICE == "fineweb-edu":
    ds = load_dataset('HuggingFaceFW/fineweb-edu', split='train', streaming=True)
    MAX_CHARS = 20_000_000
elif DATASET_CHOICE == "wikipedia":
    ds = load_dataset('wikimedia/wikipedia', '20231101.en', split='train', streaming=True)
    MAX_CHARS = 10_000_000
else:
    raise ValueError(f"Unknown dataset: {DATASET_CHOICE}")

# Stream text
def stream_text(dataset, max_chars):
    text_parts = []
    total = 0
    for example in dataset:
        text_parts.append(example['text'])
        total += len(example['text'])
        if total >= max_chars:
            break
    return ' '.join(text_parts)

print(f"Streaming {DATASET_CHOICE} (max {MAX_CHARS:,} chars)...")
text = stream_text(ds, MAX_CHARS)
print(f"Collected {len(text):,} characters")

In [ ]:
# ── Constants ──────────────────────────────────────────────────
CHARS = string.printable
VOCAB_SIZE = len(CHARS)
CHAR_TO_IDX = {c: i for i, c in enumerate(CHARS)}
IDX_TO_CHAR = {i: c for i, c in enumerate(CHARS)}
VOCAB_SIZE = len(CHARS)

CHUNK_SIZE = 8
LATENT_DIM = 128
CODEBOOK_SIZE = 256
NUM_SUBSPACES = 4
SUBSPACE_ENTRIES = 64
EMBED_DIM = 32
HIDDEN_DIM = 128

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Model Definition ──────────────────────────────────────────
class CharEmbeddingEncoder(nn.Module):
    def __init__(self, chunk_size, latent_dim, embed_dim=32, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, embed_dim)
        self.net = nn.Sequential(
            nn.Linear(chunk_size * embed_dim, hidden_dim),
            nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, latent_dim)
        )
    def forward(self, x):
        idx = torch.argmax(x, dim=-1)
        emb = self.embedding(idx)
        return self.net(emb.view(emb.size(0), -1))

class CharEmbeddingDecoder(nn.Module):
    def __init__(self, chunk_size, latent_dim, embed_dim=32, hidden_dim=128):
        super().__init__()
        self.chunk_size = chunk_size
        self.embed_dim = embed_dim
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, chunk_size * embed_dim)
        )
        self.out = nn.Linear(embed_dim, VOCAB_SIZE)
    def forward(self, z):
        x = self.net(z).view(-1, chunk_size, embed_dim)
        return self.out(x).view(-1, VOCAB_SIZE)

class PQVectorQuantizer(nn.Module):
    def __init__(self, latent_dim, num_subspaces, subspace_entries, beta=0.25):
        super().__init__()
        assert latent_dim % num_subspaces == 0
        self.latent_dim = latent_dim
        self.num_subspaces = num_subspaces
        self.subspace_dim = latent_dim // num_subspaces
        self.subspace_entries = subspace_entries
        self.beta = beta
        self.codebooks = nn.ModuleList([
            nn.Embedding(subspace_entries, latent_dim // num_subspaces)
            for _ in range(num_subspaces)
        ])
        for emb in self.codebooks:
            nn.init.uniform_(emb.weight, -1.0 / subspace_entries, 1.0 / subspace_entries)
        self.register_buffer('usage_counts', torch.zeros(num_subspaces, subspace_entries))

    def forward(self, z):
        B = z.size(0)
        z_reshaped = z.view(B, self.num_subspaces, self.subspace_dim)
        all_indices, all_quantized = [], []
        total_embed_loss = total_commit_loss = 0.0
        for i, codebook in enumerate(self.codebooks):
            z_sub = z_reshaped[:, i, :]
            distances = (z_sub ** 2).sum(1, keepdim=True) + \
                       (codebook.weight ** 2).sum(1) - \
                       2 * z_sub @ codebook.weight.t()
            indices = torch.argmin(distances, dim=1)
            quantized_sub = codebook(indices)
            all_indices.append(indices)
            all_quantized.append(quantized_sub)
            self.usage_counts[i] += torch.bincount(indices, minlength=self.subspace_entries)
            total_commit_loss += self.beta * ((quantized_sub.detach() - z_sub) ** 2).mean()
            total_embed_loss += ((quantized_sub - z_sub.detach()) ** 2).mean()
        return (torch.cat(all_quantized, dim=1),
                torch.stack(all_indices, dim=1),
                total_embed_loss / self.num_subspaces,
                total_commit_loss / self.num_subspaces)

    @torch.no_grad()
    def restart_dead_codes(self, z):
        B = z.size(0)
        z_reshaped = z.view(B, self.num_subspaces, self.subspace_dim)
        for i, codebook in enumerate(self.codebooks):
            z_sub = z_reshaped[:, i, :]
            distances = (z_sub ** 2).sum(1, keepdim=True) + \
                       (codebook.weight ** 2).sum(1) - \
                       2 * z_sub @ codebook.weight.t()
            indices = torch.argmin(distances, dim=1)
            usage = torch.bincount(indices, minlength=self.subspace_entries)
            dead_mask = usage == 0
            if dead_mask.any():
                num_dead = dead_mask.sum().item()
                rand_idx = torch.randint(0, B, (num_dead,), device=z.device)
                codebook.weight.data[dead_mask] = z_sub[rand_idx].detach()
                print(f"  [Restart] Subspace {i}: reset {num_dead} dead codes")

class VQVAE(nn.Module):
    def __init__(self, chunk_size=8, latent_dim=128, codebook_size=256, num_subspaces=4):
        super().__init__()
        self.chunk_size = chunk_size
        self.latent_dim = latent_dim
        self.codebook_size = codebook_size
        self.num_subspaces = num_subspaces
        
        self.encoder = CharEmbeddingEncoder(chunk_size, latent_dim)
        self.quantizer = PQVectorQuantizer(latent_dim, num_subspaces, codebook_size // num_subspaces)
        self.decoder = CharEmbeddingDecoder(chunk_size, latent_dim)

    def forward(self, x, restart_dead=False):
        idx = torch.argmax(x, dim=-1)
        emb = self.encoder.embedding(idx)
        z = self.encoder.net(emb.view(emb.size(0), -1))
        quantized, indices, embed_loss, commit_loss = self.quantizer(z)
        if restart_dead and self.training:
            with torch.no_grad(): self.quantizer.restart_dead_codes(z)
        recon_logits = self.decoder(quantized)
        return recon_logits, indices, embed_loss, commit_loss

    def encode(self, x):
        self.eval()
        with torch.no_grad():
            _, indices, _, _ = self.forward(x)
        return indices.cpu().numpy()

    def decode_codes(self, codes):
        self.eval()
        with torch.no_grad():
            indices = torch.tensor(codes, dtype=torch.long, device=next(self.parameters()).device)
            if indices.ndim == 1:
                quantized = self.quantizer.codebooks[0](indices)
                quantized = quantized.repeat(1, self.quantizer.num_subspaces)
            else:
                quantized = torch.cat([cb(indices[:, i]) for i, cb in enumerate(self.quantizer.codebooks)], dim=1)
            recon = torch.argmax(self.decoder(quantized), dim=-1)
        return recon.cpu().numpy()

In [ ]:
# ── Helpers ───────────────────────────────────────────────────
CHARS = string.printable
VOCAB_SIZE = len(CHARS)
CHAR_TO_IDX = {c: i for i, c in enumerate(CHARS)}
IDX_TO_CHAR = {i: c for i, c in enumerate(CHARS)}
VOCAB_SIZE = len(CHARS)

CHUNK_SIZE = 8
LATENT_DIM = 128
CODEBOOK_SIZE = 256
NUM_SUBSPACES = 4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def text_to_chunks(text, chunk_size):
    chunks = []
    text = text[:len(text) // chunk_size * chunk_size]
    for i in range(0, len(text), chunk_size):
        chunk = np.zeros((chunk_size, VOCAB_SIZE), dtype=np.float32)
        for j, c in enumerate(text[i:i+chunk_size]):
            if c in CHAR_TO_IDX:
                chunk[j, CHAR_TO_IDX[c]] = 1.0
        chunks.append(chunk)
    return chunks

def reconstruction_error(original, reconstructed):
    original_idx = np.argmax(original, axis=-1)
    return float(np.sum(original_idx == reconstructed)) / original_idx.size

In [ ]:
# ── Prepare Data ──────────────────────────────────────────────
from datasets import load_dataset

DATASET_CHOICE = "wikitext"  # Change to: c4, fineweb, pile, fineweb-edu, wikipedia

if DATASET_CHOICE == "wikitext":
    ds = load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1', split='train', streaming=True)
    MAX_CHARS = 5_000_000
elif DATASET_CHOICE == "c4":
    ds = load_dataset('allenai/c4', 'en', split='train', streaming=True)
    MAX_CHARS = 10_000_000
elif DATASET_CHOICE == "fineweb":
    ds = load_dataset('HuggingFaceFW/fineweb', split='train', streaming=True)
    MAX_CHARS = 20_000_000
elif DATASET_CHOICE == "pile":
    ds = load_dataset('EleutherAI/pile', split='train', streaming=True)
    MAX_CHARS = 10_000_000
elif DATASET_CHOICE == "fineweb-edu":
    ds = load_dataset('HuggingFaceFW/fineweb-edu', split='train', streaming=True)
    MAX_CHARS = 20_000_000
elif DATASET_CHOICE == "wikipedia":
    ds = load_dataset('wikimedia/wikipedia', '20231101.en', split='train', streaming=True)
    MAX_CHARS = 10_000_000
else:
    raise ValueError(f"Unknown dataset: {DATASET_CHOICE}")

def stream_text(dataset, max_chars):
    text_parts = []
    total = 0
    for example in dataset:
        text_parts.append(example['text'])
        total += len(example['text'])
        if total >= max_chars:
            break
    return ' '.join(text_parts)

print(f"Streaming {DATASET_CHOICE} (max {MAX_CHARS:,} chars)...")
text = stream_text(ds, MAX_CHARS)
print(f"Collected {len(text):,} characters")

chunks = text_to_chunks(text, CHUNK_SIZE)
dataset = torch.tensor(np.stack(chunks), dtype=torch.float32, device=DEVICE)
print(f"Dataset: {len(chunks)} chunks, {dataset.shape}")

In [ ]:
# ── Model & Optimizer ────────────────────────────────────────
model = VQVAE(CHUNK_SIZE, 128, 256, 4).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=400, eta_min=1e-6)

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Training on {DEVICE}")

In [ ]:
# ── Training Loop ────────────────────────────────────────────
EPOCHS = 400
BATCH_SIZE = 512
loss_history = []

for epoch in range(400):
    model.train()
    perm = torch.randperm(len(dataset), device=DEVICE)
    epoch_recon = epoch_embed = epoch_commit = 0.0
    
    for i in range(0, len(dataset), 512):
        batch_idx = perm[i:i+512]
        x = dataset[batch_idx]
        restart_dead = (epoch > 0) and (epoch % 50 == 0) and (i == 0)
        recon_logits, _, embed_loss, commit_loss = model(x, restart_dead=restart_dead)
        recon_loss = nn.CrossEntropyLoss(label_smoothing=0.1)(
            recon_logits.view(-1, VOCAB_SIZE), x.argmax(-1).view(-1))
        loss = recon_loss + 0 + 0  # losses already in embed_loss/commit_loss
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        epoch_recon += recon_loss.item() * len(batch_idx)
        epoch_embed += 0 * len(batch_idx)
        epoch_commit += 0 * len(batch_idx)
    
    scheduler.step()
    epoch_recon /= len(dataset)
    epoch_embed /= len(dataset)
    epoch_commit /= len(dataset)
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d} | Recon: {epoch_recon:.4f}")

print("Training complete!")

In [ ]:
# Save model
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {'chunk_size': CHUNK_SIZE, 'latent_dim': 128, 'codebook_size': 256, 'num_subspaces': 4},
}, 'stage1_vqvae.pt')
print("Saved stage1_vqvae.pt")

In [ ]:
# Evaluation
model.eval()
with torch.no_grad():
    codes = model.encode(dataset)
    recon = model.decode_codes(codes)
    original = torch.tensor(np.stack(text_to_chunks(text, CHUNK_SIZE)), device=DEVICE)
    rec_error = reconstruction_error(original.cpu().numpy(), recon.reshape(-1, CHUNK_SIZE))
    util = len(np.unique(codes)) / 256 * 100
    print(f"Recon error: {rec_error:.4f} | Util: {util:.1f}% | Unique codes: {len(np.unique(codes))}")

# Save for Stage 2
np.save("stage1_codes.npy", codes)
np.save("stage1_recon.npy", model.decode_codes(model.encode(
    torch.tensor(np.stack(text_to_chunks(text, CHUNK_SIZE)), dtype=torch.float32, device=DEVICE))))
print("Saved stage1_codes.npy & stage1_recon.npy")